# Self-Supervised Pretraining for Network Traffic

Experiment notebook -- equivalent code to `scripts/run_pipeline.py`. Pretrain a small transformer encoder on **unlabeled** traffic (masked-feature reconstruction + masked-view contrastive), then compare fine-tuned accuracy vs **from-scratch** at 5% / 10% / 20% of labels.

## 1. Setup + generate synthetic traffic

Flows are sampled from 8 traffic archetypes (web, dns, ssh, mail, video, portscan, bruteforce, exfil) over 16 realistic features. The labeled set is a stratified subset of the unlabeled pool; the test set is held out.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import torch

from sstraffic import make_dataset, pretrain_encoder, train_classifier, embeddings_np, tsne_scatter, accuracy_curve_plot, loss_plot

ds = make_dataset(n_unlabeled=4000, labeled_per_class=100, n_test=1600, seed=0)
print("unlabeled:", ds.unlabeled.shape, "| labeled:", ds.labeled_X.shape, "| test:", ds.test_X.shape)
print("features:", ds.feature_names)
print("archetypes:", ds.archetype_names)

## 2. Pretrain the encoder

Mask ~30% of feature dimensions per sample and regress the raw values back (primary objective); a masked-view InfoNCE term sharpens the sample embedding.

In [ ]:
encoder, hist = pretrain_encoder(
    ds.unlabeled, n_features=ds.unlabeled.shape[1],
    epochs=30, batch_size=256, mask_frac=0.3, lr=2e-3, seed=0,
)
print("final masked-MSE:", round(hist[-1]["rec"], 4))
loss_plot([h["rec"] for h in hist], "../results/figures/pretraining_loss.png")
print("saved ../results/figures/pretraining_loss.png")

## 3. Fine-tune sweep: pretrained vs from-scratch (3 seeds averaged)

The only difference between the two arms is the encoder initialization.

In [ ]:
init_state = {k: v.clone() for k, v in encoder.state_dict().items()}

def stratified_subset(X, y, frac, rng):
    idx = []
    for c in range(ds.n_classes):
        ci = np.where(y == c)[0]
        k = max(1, int(round(frac * len(ci))))
        idx.append(rng.choice(ci, size=k, replace=False))
    idx = np.concatenate(idx)
    return X[idx], y[idx]

fractions = [0.05, 0.10, 0.20]
pre_accs, scr_accs = [], []
for fi, frac in enumerate(fractions):
    Xs, ys = stratified_subset(ds.labeled_X, ds.labeled_y, frac, np.random.default_rng(1000 * (fi + 1)))
    pre, scr = [], []
    for s in range(3):
        pre.append(train_classifier(Xs, ys, ds.test_X, ds.test_y, ds.n_classes, ds.unlabeled.shape[1],
                                    init_state=init_state, epochs=30, seed=s)["test_acc"])
        scr.append(train_classifier(Xs, ys, ds.test_X, ds.test_y, ds.n_classes, ds.unlabeled.shape[1],
                                    init_state=None, epochs=30, seed=100 + s)["test_acc"])
    p, q = float(np.mean(pre)), float(np.mean(scr))
    pre_accs.append(p); scr_accs.append(q)
    print(f"{frac*100:>3.0f}% labels (n={len(Xs):>3}): pretrained={p*100:5.1f}%  scratch={q*100:5.1f}%  gain={100*(p-q):+5.1f} pts")

accuracy_curve_plot(fractions, pre_accs, scr_accs, "../results/figures/label_fraction_accuracy.png")
print("saved ../results/figures/label_fraction_accuracy.png")

## 4. t-SNE of the pretrained encoder embeddings

Colored by traffic archetype (attack classes in red). The unlabeled encoder discovers traffic structure without labels.

In [ ]:
emb = embeddings_np(encoder, ds.unlabeled, device="cpu")
tsne_scatter(emb, ds.unlabeled_y, ds.archetype_names, "../results/figures/tsne_clusters.png")
print("saved ../results/figures/tsne_clusters.png")

## 5. Summary

- Pretraining helps most when labels are scarce: large gain at 5% labels, shrinking as labels grow.
- The t-SNE plot shows the encoder separated normal traffic (web, dns, ssh, mail, video) from attack flows (portscan, bruteforce, exfil) with zero labels.
- Run `scripts/run_pipeline.py` end-to-end to regenerate `results/metrics.md` with real numbers.